# Milvus的基本使用

# 1、DDL操作

## 1.1 数据库相关操作

### ① 查看数据库

举例1：操作客户端

In [3]:

from pymilvus import MilvusClient

client = MilvusClient("http://localhost:19530")

举例2：列出所有数据库

In [7]:
existed_databases = client.list_databases()

for db in existed_databases:
    print(db)

default
rag_demo


### ② 创建数据库

In [6]:
db_name = "rag_demo"

if db_name not in existed_databases:
    client.create_database(db_name=db_name)

### ③ 删除数据库

如果数据库下有Collection则无法删除，需要先删除它的所有Collection才能删除Database

In [6]:

client.drop_database(db_name = db_name)

## 1.2 Collection相关操作

### ① 切换数据库

In [8]:
client.use_database(db_name=db_name)

### ② 查看数据库下的collections

In [11]:
collections = client.list_collections()

for coll in collections:
    print(coll)

docs


### ③ 创建collection

In [10]:
collection_name = "docs"

client.create_collection(
    collection_name=collection_name,
    dimension=1024,
    metric_type="COSINE"
)

### ④ 删除collection

In [14]:
client.drop_collection(collection_name=collection_name)

# 2、DML操作

## 2.1 嵌入模型的初始化

In [25]:
from langchain.embeddings import init_embeddings
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# 初始化嵌入模型
embed_model = init_embeddings(
	model="openai:embedding-2",
	api_key=os.getenv("ZHIPUAI_EMBEDDING_API_KEY"),
	base_url=os.getenv("ZHIPUAI_EMBEDDING_BASE_URL"),
)

## 2.2 准备collection

### ① 创建collection

In [26]:
collection_name = "docs"

client.create_collection(
    collection_name=collection_name,
    dimension=1024,
    metric_type="COSINE"
)

### ② 查看collection元数据

In [27]:
from rich import print as rprint

metadata = client.describe_collection(collection_name=collection_name)

rprint(metadata)

{
    'collection_name': 'docs',
    'auto_id': False,
    'num_shards': 1,
    'description': '',
    'fields': [
        {
            'field_id': 100,
            'name': 'id',
            'description': '',
            'type': <DataType.INT64: 5>,
            'params': {},
            'is_primary': True
        },
        {
            'field_id': 101,
            'name': 'vector',
            'description': '',
            'type': <DataType.FLOAT_VECTOR: 101>,
            'params': {'dim': 1024}
        }
    ],
    'functions': [],
    'aliases': [],
    'collection_id': 467879810829651595,
    'consistency_level': 2,
    'properties': {'timezone': 'UTC'},
    'num_partitions': 1,
    'enable_dynamic_field': True,
    'enable_namespace': False,
    'created_timestamp': 467879949835960338,
    'update_timestamp': 467879949835960338
}

## 2.3 准备数据

### ① 准备原始数据

In [28]:
# 准备测试数据
texts = [
    "LangChain 是一个用于构建 LLM 应用的开发框架。",
    "Milvus 是一个适合 AI 应用的向量数据库。",
    "RAG 的核心是先检索相关知识，再让大模型生成答案。",
    "Docker Desktop 可以方便地在本地运行 Milvus Standalone。"
]

### ② 生成嵌入向量

In [30]:
vectors = embed_model.embed_documents(texts)

### ③ 查看生成的嵌入向量

In [31]:
print(len(vectors))

print(len(vectors[0]))

print(vectors[0][:5])

4
1024
[-0.0026002603, 0.0073529813, 0.04261343, 0.045320854, 0.037619848]


### ④ 封装为可以插入的数据格式

In [32]:
data = [
    {
        "id" : i,
        "vector" : vectors[i],
        "text" : texts[i],
        "source" : "demo"
    } for i in range(len(texts))
]

## 2.4 写入数据

### ① 插入数据

In [33]:
insert_res = client.upsert(
    collection_name=collection_name,
    data=data,
)

print("insert result : ",insert_res)

insert result :  {'upsert_count': 4, 'ids': [0, 1, 2, 3]}


### ② 手动flush

Milvus不会第一时间将数据落盘，要看到写入效果，我们手动flush，将数据刷写到磁盘

In [34]:
client.flush(collection_name=collection_name)

### ③ 查看collection统计信息

In [35]:
stats = client.get_collection_stats(collection_name=collection_name)

print("stats : ",stats)

stats :  {'row_count': 4}


# 3、DQL操作

## 3.1 扫描数据

In [36]:

iterator = client.query_iterator(
    collection_name=collection_name,
    filter="",
    output_fields=["*"]
)

i = 0
while True:

    rows = iterator.next()

    if not rows:
        break

    for row in rows:
        print(f"第{i + 1}条数据：")
        # print(row)

        print(f"id : {row["id"]},vector = {row["vector"][:5]},text = {row["text"]},source = {row["source"]}")

        i += 1

iterator.close()

第1条数据：
id : 0,vector = [-0.0026002603117376566, 0.007352981250733137, 0.0426134318113327, 0.04532085359096527, 0.03761984780430794],text = LangChain 是一个用于构建 LLM 应用的开发框架。,source = demo
第2条数据：
id : 1,vector = [-0.028767047449946404, -0.01970093697309494, 0.048395249992609024, 0.028413202613592148, 0.04291243106126785],text = Milvus 是一个适合 AI 应用的向量数据库。,source = demo
第3条数据：
id : 2,vector = [-0.005844089202582836, 0.028109155595302582, 0.015837395563721657, 0.036579400300979614, 0.012819184921681881],text = RAG 的核心是先检索相关知识，再让大模型生成答案。,source = demo
第4条数据：
id : 3,vector = [0.0017661118181422353, 0.004658818710595369, 0.02571912482380867, 0.02016250044107437, 0.023687761276960373],text = Docker Desktop 可以方便地在本地运行 Milvus Standalone。,source = demo


## 3.2 通过主键查询数据

In [37]:

res = client.get(
    collection_name=collection_name,
    ids=[0,1,2]
)

print(len(res))

for i in range(len(res)):
    print(f"第{i + 1}条数据：")
    print(f"id : {res[i]["id"]},vector = {res[i]["vector"][:5]},text = {res[i]["text"]},source = {res[i]["source"]}")
    # print(res[i])


3
第1条数据：
id : 0,vector = [-0.0026002603117376566, 0.007352981250733137, 0.0426134318113327, 0.04532085359096527, 0.03761984780430794],text = LangChain 是一个用于构建 LLM 应用的开发框架。,source = demo
第2条数据：
id : 1,vector = [-0.028767047449946404, -0.01970093697309494, 0.048395249992609024, 0.028413202613592148, 0.04291243106126785],text = Milvus 是一个适合 AI 应用的向量数据库。,source = demo
第3条数据：
id : 2,vector = [-0.005844089202582836, 0.028109155595302582, 0.015837395563721657, 0.036579400300979614, 0.012819184921681881],text = RAG 的核心是先检索相关知识，再让大模型生成答案。,source = demo


## 3.3 相似度检索

### ① 准备查询嵌入

In [49]:
# 相似度检索
query = "什么是Milvus？"
query_vector = embed_model.embed_query(query)

### ② 检索

In [50]:
results = client.search(
    collection_name=collection_name,
    data=[query_vector],
    limit=3,
    output_fields=["text","source","id"]
)

for res in results[0]:
    print(res)

{'id': 0, 'distance': 0.49188581109046936, 'entity': {'id': 0, 'text': 'LangChain 是一个用于构建 LLM 应用的开发框架。', 'source': 'demo'}}
{'id': 3, 'distance': 0.4822562336921692, 'entity': {'id': 3, 'text': 'Docker Desktop 可以方便地在本地运行 Milvus Standalone。', 'source': 'demo'}}
{'id': 1, 'distance': 0.4780961871147156, 'entity': {'id': 1, 'text': 'Milvus 是一个适合 AI 应用的向量数据库。', 'source': 'demo'}}
